# Trend Inflation

This example is a replication of the univariate state space model suggested by (Stock &
Watson, 2016) using GeneralisedFilters to define a heirarchical model for use in Rao-
Blackwellised particle filtering.

Install dependencies so the notebook runs on a fresh Colab runtime. GeneralisedFilters
and SSMProblems are added from the repo's main branch (registered versions may be too
old, e.g. lack ReferenceTrajectory):

In [ ]:
import Pkg, Downloads
Downloads.download(
    "https://raw.githubusercontent.com/TuringLang/SSMProblems.jl/main/GeneralisedFilters/examples/trend-inflation/Project.toml",
    "Project.toml",
)
Pkg.activate(".")
Pkg.add([
    Pkg.PackageSpec(; url="https://github.com/TuringLang/SSMProblems.jl", subdir="GeneralisedFilters", rev="main"),
    Pkg.PackageSpec(; url="https://github.com/TuringLang/SSMProblems.jl", subdir="SSMProblems", rev="main"),
])
Pkg.instantiate()

using GeneralisedFilters
using Distributions
using Random
using StatsBase
using LinearAlgebra
using StaticArrays

const GF = GeneralisedFilters



Download the helper script and data so the notebook is self-contained on Colab:

In [ ]:
using Downloads
INFL_URL = "https://raw.githubusercontent.com/TuringLang/SSMProblems.jl/main/GeneralisedFilters/examples/trend-inflation"
for f in ("utilities.jl", "data.csv")
    isfile(f) || Downloads.download("$(INFL_URL)/$(f)", f)
end
INFL_PATH = pwd()
include(joinpath(INFL_PATH, "utilities.jl"))

## Model Definition

We begin by defining the local level trend model, a linear Gaussian model with a weakly
stationary random walk component. The dynamics of which are as follows:

$$
\begin{aligned}
    y_{t} &= x_{t} + \eta_{t} \\
    x_{t+1} &= x_{t} + \varepsilon_{t}
\end{aligned}
$$

However, this model is not enough to capture trend dynamics when faced with structural
breaks. (Stock & Watson, 2007) suggest adding a stochastic volatiltiy component, defined
like so:

$$
\begin{aligned}
    \log \sigma_{\eta, t+1} = \log \sigma_{\eta, t} + \nu_{\eta, t} \\
    \log \sigma_{\varepsilon, t+1} = \log \sigma_{\varepsilon, t} + \nu_{\varepsilon, t}
\end{aligned}
$$

where $\nu_{z,t} \sim N(0, \gamma)$ for $z \in \{ \varepsilon, \eta \}$.

Using `GeneralisedFilters`, we can construct a heirarchical version of this model such
that the local level trend component is conditionally linear Gaussian on the volatility
draws.

#### Stochastic Volatility Process

We begin by defining the non-linear dynamics, which aren't conditioned contemporaneous
states. Since these processes are traditionally non-linear/non-Gaussian we use the
process interface to define the stochastic volatility components.

In [ ]:
struct StochasticVolatilityPrior{T<:Real} <: StatePrior end

In [ ]:
function GF.distribution(prior::StochasticVolatilityPrior{T}) where {T}
    return product_distribution(Normal(zero(T), T(1)), Normal(zero(T), T(1)))
end

For the dynamics, instead of using the `GF.distribution` utility, we only define
the `simulate` method, which is sufficient for the RBPF.

In [ ]:
struct StochasticVolatility{ΓT<:AbstractVector} <: LatentDynamics
    γ::ΓT
end

In [ ]:
function GF.simulate(
    rng::AbstractRNG, proc::StochasticVolatility, step::Integer, state::AbstractVector{T}
) where {T<:Real}
    new_state = deepcopy(state)
    new_state[1:2] += proc.γ .* randn(rng, T, 2)
    return new_state
end

#### Local Level Trend Process

Resolve small Gaussian atoms from the current outer state. Static arrays preserve
the one-dimensional inner state throughout Kalman filtering.

In [ ]:
function local_level(ctx)
    return LinearGaussianDynamics(
        @SMatrix([1.0;;]), @SVector([0.0]), SMatrix{1,1}(exp(ctx.x_new[1]))
    )
end
function simple_observation(ctx)
    return LinearGaussianObservation(
        @SMatrix([1.0;;]), @SVector([0.0]), SMatrix{1,1}(exp(ctx.x[2]))
    )
end

### Unobserved Components with Stochastic Volatility

The state space model suggested by (Stock & Watson, 2007) can be constructed with the
following method:

In [ ]:
function UCSV(γ::T) where {T<:Real}
    stoch_vol_prior = StochasticVolatilityPrior{T}()
    stoch_vol_process = StochasticVolatility(fill(γ, 2))

    return StateSpaceModel(
        stoch_vol_prior,
        stoch_vol_process,
        GaussianPrior(@SVector([0.0]), @SMatrix([100.0;;])),
        local_level,
        simple_observation,
    )
end;

For plotting, an explicit filtering loop records ancestry after each step.

In [ ]:
rng = MersenneTwister(1234);
states, ll, tree = filter_with_ancestry(
    rng,
    UCSV(0.2),
    RBPF(BF(2^12), KalmanFilter()),
    [SVector(pce) for pce in fred_data.value],
);

The tree stores particle ancestry which we can use to approximate
the smoothed series without an additional backwards pass. We can convert this data
structure to a human readable array by using `GeneralisedFilters.get_ancestry` and then
take the mean path by passing a custom function.

In [ ]:
trends, volatilities = mean_path(GF.get_ancestry(tree), states);
plot_ucsv(trends[1, :], eachrow(volatilities), fred_data)

#### Outlier Adjustments

For additional robustness, (Stock & Watson, 2016) account for one-time measurement shocks
and suggest an alteration in the observation equation, where

$$
\eta_{t} \sim N(0, s_{t} \cdot \sigma_{\eta, t}^2) \quad \quad s_{t} \sim \begin{cases}
U(2,10) & \text{ with probability } p \\
\delta(1) & \text{ with probability } 1 - p
\end{cases}
$$

The prior is the same as before, but with additional state which we can assume will always
be 1; using the `Distributions` interface this is just `Dirac(1)`

In [ ]:
struct OutlierAdjustedVolatilityPrior{T<:Real} <: StatePrior end

In [ ]:
function GF.distribution(prior::OutlierAdjustedVolatilityPrior{T}) where {T}
    return product_distribution(Normal(zero(T), T(1)), Normal(zero(T), T(1)), Dirac(one(T)))
end

In terms of the model definition, we can construct a separate `LatentDynamics` which
contains the same volatility process as before, but with the respective draw in the third
component.

In [ ]:
struct OutlierAdjustedVolatility{ΓT} <: LatentDynamics
    volatility::StochasticVolatility{ΓT}
    switch_dist::Bernoulli
    outlier_dist::Uniform
end

The simulation then calls the volatility process, and computes the outlier term in the
third state

In [ ]:
function GF.simulate(
    rng::AbstractRNG,
    proc::OutlierAdjustedVolatility,
    step::Integer,
    state::AbstractVector{T},
) where {T<:Real}
    new_state = GF.simulate(rng, proc.volatility, step, state)
    new_state[3] = rand(rng, proc.switch_dist) ? rand(rng, proc.outlier_dist) : one(T)
    return new_state
end

For the observation process, we define a new object where $R$ is dependent on both the
measurement volatility as well as this outlier adjustment coefficient.

In [ ]:
function outlier_observation(ctx)
    return LinearGaussianObservation(
        @SMatrix([1.0;;]), @SVector([0.0]), SMatrix{1,1}(ctx.x[3] * exp(ctx.x[2]))
    )
end

### Outlier Adjusted UCSV

The state space model suggested by (Stock & Watson, 2007) can be constructed with the
following method:

In [ ]:
function UCSVO(γ::T, prob::T) where {T<:Real}
    stoch_vol_prior = OutlierAdjustedVolatilityPrior{T}()
    stoch_vol_process = OutlierAdjustedVolatility(
        StochasticVolatility(fill(γ, 2)), Bernoulli(prob), Uniform{T}(2, 10)
    )

    return StateSpaceModel(
        stoch_vol_prior,
        stoch_vol_process,
        GaussianPrior(@SVector([0.0]), @SMatrix([100.0;;])),
        local_level,
        outlier_observation,
    )
end;

We then repeat the same experiment, this time with an outlier probability of $p = 0.05$

In [ ]:
rng = MersenneTwister(1234);
states, ll, tree = filter_with_ancestry(
    rng,
    UCSVO(0.2, 0.05),
    RBPF(BF(2^12), KalmanFilter()),
    [SVector(pce) for pce in fred_data.value],
);

this process is identical to the last, except with an additional `volatilities` state
which captures the outlier distance. We omit this feature in the plots, but the impact is
clear when comparing the maximum transitory noise around the GFC.

In [ ]:
trends, volatilities = mean_path(GF.get_ancestry(tree), states);
plot_ucsv(trends[1, :], eachrow(volatilities), fred_data)

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*